# Transformer de snapshot · diario de experimentos (Pablo · A)

**Qué es:** `src/m_nn.py`, una red que mira **todas las tareas de un snapshot a la vez** (self-attention entre tareas)
y da una puntuación a cada una; el softmax sobre la lista dice cuál es la tarea en la que trabaja el controlador.

**Cómo usar este notebook**
1. Ejecuta *Preparación* una vez (carga los datos, ~30 s).
2. Cada bloque de experimentos tiene: **hipótesis** → **variantes** (se cambia UNA cosa respecto a la base) → **tabla**.
3. Escribe la conclusión en la celda *Conclusión* de cada bloque antes de pasar al siguiente.
4. Lo que gane se combina en `MEJOR`, se confirma con 3 semillas y se guarda para el stacking.

**Reglas que respeta el modelo** (`CLAUDE.md` / `EQUIPO.md`)
- Cada snapshot se predice **solo con sus propias filas**: nada de snapshots anteriores o posteriores.
- CV con `Data/folds.csv` (5 folds por controlador; cada sesión queda entera en un fold).
- Early stopping con 3 controladores apartados *dentro* del fold de entrenamiento (nunca mira el fold de validación).
- No usa `sample_id, session, atco, scenario, timestamp, task_name, row_index`.

**Cómo leer las cifras**
- `cv` = top-3 hit rate out-of-fold sobre train+validation. Referencia: LightGBM ≈ 0.655 (con los folds antiguos; pendiente de recalcular con `folds.csv`).
- Las pruebas rápidas usan **1 semilla**: diferencias < ~0.005 son ruido. Lo que parezca ganar se confirma con 3.
- `min_atco` = nota del peor controlador. Nos importa: el privado es **un solo controlador** (023) que no está en train.
- **Decisión a las 21:00:** si la CV queda a más de ~3 puntos del GBM, la red pasa a modelo secundario del ensemble.

## Preparación

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import m_nn
print("device:", m_nn.DEV)

data = m_nn.build_data()          # una sola vez; todas las pruebas lo reutilizan
LGB_CV = 0.655                    # referencia del equipo (actualizar cuando Adri la recalcule con folds.csv)

In [ ]:
# Configuración base = la de EQUIPO.md. Cada experimento cambia UNA de estas claves.
BASE = dict(d=64, heads=4, layers=3, drop=0.2, ls=0.1, lr=1e-3, wd=0.05, bs=128, epochs=40, patience=6, noise=0.0)


def probar(nombre, motivo, semillas=1, guardar=False, **cambios):
    """Lanza la CV con BASE + cambios. Queda apuntado en experiments/nn_runs.csv con su motivo.
    guardar=True genera oof/, preds/ y submissions/ (solo para la versión que va al stacking)."""
    cfg = {**BASE, **cambios}
    args = ["--name", nombre, "--tag", motivo] + (["--quick"] if not guardar else [])
    for k, v in cfg.items():
        args += [f"--{k.replace('_', '-')}", str(v)]
    a = m_nn.parse(args)
    a.seeds = semillas
    return m_nn.run(a, data)


def lanzar(bloque, variantes, semillas=1):
    """variantes: lista de (nombre, motivo, {cambios}). Siempre incluye la base para comparar en igualdad."""
    res = [("base", "referencia", probar(f"{bloque}/base", "referencia", semillas))]
    for nombre, motivo, cambios in variantes:
        res.append((nombre, motivo, probar(f"{bloque}/{nombre}", motivo, semillas, **cambios)))
    return tabla_bloque(res)


def tabla_bloque(res):
    t = pd.DataFrame(res, columns=["variante", "por qué", "cv"])
    ref = t.loc[t.variante == "base", "cv"].iloc[0]
    t["Δ vs base"] = t.cv - ref
    t["Δ vs LGB"] = t.cv - LGB_CV
    t["veredicto"] = t["Δ vs base"].map(lambda x: "—" if abs(x) < 1e-9 else ("mejora" if x > 0.005 else
                                                   ("empeora" if x < -0.005 else "ruido")))
    return (t.sort_values("cv", ascending=False).style
            .format({"cv": "{:.4f}", "Δ vs base": "{:+.4f}", "Δ vs LGB": "{:+.4f}"})
            .background_gradient(subset=["Δ vs base"], cmap="RdYlGn", vmin=-0.02, vmax=0.02).hide(axis="index"))


def historial(filtro=""):
    """Todo lo probado hasta ahora (también desde la terminal), ordenado por CV."""
    r = pd.read_csv(m_nn.EXP / "nn_runs.csv")
    r = r[r.name.str.contains(filtro, regex=False)] if filtro else r
    cols = ["when", "name", "tag", "cv", "min_atco", "seeds", "d", "layers", "drop", "wd", "noise", "ls", "lr", "bs",
            "best_epochs", "secs"]
    return r[[c for c in cols if c in r]].sort_values("cv", ascending=False).reset_index(drop=True)

## 0 · Punto de partida

Configuración de `EQUIPO.md` con 3 semillas y `folds.csv`: **CV 0.609** (folds 0.654 / 0.590 / 0.615 / 0.593 / 0.597,
peor controlador 0.543). Queda a ~4.5 puntos del LightGBM.

**Diagnóstico:** las mejores épocas salen casi siempre entre la 3 y la 7 de 40 → el modelo aprende rápido los
controladores de entrenamiento y enseguida empieza a **sobreajustar**. Por eso lo primero es regularizar.

In [ ]:
historial()

## 1 · Regularización

**Hipótesis:** si el problema es sobreajuste, ponerle trabas al modelo debería subir la CV y retrasar la mejor época.

| Cambio | Por qué podría ayudar |
|---|---|
| `drop` 0.3 / 0.4 | apaga neuronas al azar: no puede depender de una sola señal |
| `wd` 0.1 / 0.2 | penaliza pesos grandes: función más suave |
| `noise` 0.05 / 0.1 | ruido en las features al entrenar: no se fía de valores exactos (las etiquetas ya tienen ruido) |

~1 min por variante.

In [ ]:
lanzar("reg", [
    ("drop=0.3",   "más dropout contra sobreajuste",        dict(drop=0.3)),
    ("drop=0.4",   "aún más dropout",                       dict(drop=0.4)),
    ("wd=0.1",     "más weight decay",                      dict(wd=0.1)),
    ("wd=0.2",     "aún más weight decay",                  dict(wd=0.2)),
    ("noise=0.05", "ruido en features, robustez",           dict(noise=0.05)),
    ("noise=0.1",  "más ruido en features",                 dict(noise=0.1)),
])

**Conclusión bloque 1:** _(rellenar: qué mejora, qué es ruido, ¿se retrasa la mejor época?)_

## 2 · Capacidad del modelo

**Hipótesis:** con ~13 000 snapshots, un modelo más pequeño puede generalizar mejor a controladores nuevos;
uno más grande solo ayuda si el bloque 1 ha controlado el sobreajuste.

| Cambio | Por qué |
|---|---|
| `d` 32 | menos capacidad para memorizar |
| `d` 128 | más capacidad (por si la base se queda corta) |
| `layers` 2 | menos pasos de atención entre tareas |
| `layers` 4 | más interacción entre tareas (comparar entre aviones, tipos…) |

In [ ]:
lanzar("cap", [
    ("d=32",     "modelo más pequeño",         dict(d=32)),
    ("d=128",    "modelo más grande",          dict(d=128)),
    ("layers=2", "menos capas de atención",    dict(layers=2)),
    ("layers=4", "más capas de atención",      dict(layers=4)),
])

**Conclusión bloque 2:** _(rellenar)_

## 3 · Optimización y pérdida

| Cambio | Por qué |
|---|---|
| `lr` 5e-4 | pasos más pequeños: aprende más despacio y más estable |
| `lr` 2e-3 | por si la base se queda corta |
| `ls` 0.05 / 0.2 | label smoothing: las etiquetas tienen ruido y los controladores no coinciden mucho → suavizar puede ayudar |
| `bs` 64 / 256 | lotes pequeños meten más ruido en el gradiente (regulariza); grandes, más estables |

In [ ]:
lanzar("opt", [
    ("lr=5e-4", "aprendizaje más lento",      dict(lr=5e-4)),
    ("lr=2e-3", "aprendizaje más rápido",     dict(lr=2e-3)),
    ("ls=0.05", "menos label smoothing",      dict(ls=0.05)),
    ("ls=0.2",  "más label smoothing",        dict(ls=0.2)),
    ("bs=64",   "lotes pequeños",             dict(bs=64)),
    ("bs=256",  "lotes grandes",              dict(bs=256)),
])

**Conclusión bloque 3:** _(rellenar)_

## 4 · Combinar lo que ha funcionado

Pon en `MEJOR` solo los cambios con veredicto **mejora** en los bloques anteriores.
Se comprueba con **3 semillas** frente a la base con 3 semillas (0.609), porque varios cambios que ayudan por separado
no siempre suman.

In [ ]:
MEJOR = dict(
    # drop=0.3, noise=0.05, ...   <- rellenar con los ganadores
)
cv_comb = probar("comb/" + ",".join(f"{k}={v}" for k, v in MEJOR.items()), "combinación de ganadores", semillas=3, **MEJOR)
print(f"combinación: {cv_comb:.4f}   vs base 3 semillas 0.609   vs LGB {LGB_CV}")

**Conclusión bloque 4:** _(rellenar)_

## 5 · Versión para el stacking

Si la combinación mejora a la base, se genera la versión definitiva con 3 semillas: guarda `oof/`, `preds/` y
`submissions/` en el formato común. Juanma la usa en el stacking (**nadie más sube a Kaggle**).
Cambia el nombre (`nn_tf_v2`, `v3`…) para no pisar versiones anteriores.

In [ ]:
NOMBRE = "nn_tf_v2"
probar(NOMBRE, "versión para stacking: " + str(MEJOR), semillas=3, guardar=True, **MEJOR)

In [ ]:
# Comprobación con el scorer oficial sobre las muestras de validation.csv
!cd {ROOT} && {sys.executable} Data/scorer.py --labelled Data/validation.csv oof/sub_{NOMBRE}_validation.csv

## 6 · Siguientes ideas (no implementadas todavía)

- **Usar el OOF del LightGBM de Adri como feature de entrada** (variante 3 de `EQUIPO.md`). Necesita `oof/lgb_*.csv` y
  `preds/lgb_*.csv` con los mismos folds.
- **Sesgo de atención "mismo avión"**: que la atención sepa qué tareas son del mismo avión (sobre todo assume vs
  entry_coordination, `l_twin_AE`). El avión sale de `task_name`, que no se puede usar como feature: consultarlo con el equipo.
- **Si se permiten otros snapshots:** que la red vea también la lista anterior y la siguiente.

## Historial completo

In [ ]:
historial()